<div style='text-align:center; padding:40px 0 20px;'>
<h1 style="font-family:'Playfair Display',serif; font-size:36px; color:#1B2A4A; margin:0; letter-spacing:1px;">Dataset Captioner</h1>
<p style="font-family:'Inter',sans-serif; font-size:14px; color:#8B7D6B; margin-top:8px; letter-spacing:0.5px;">Image Captioning Studio for Google Colab</p>
<hr style="width:80px; border:none; border-top:1.5px solid #C5A55A; margin:16px auto;">
</div>

In [ ]:
#@title Setup Environment { display-mode: "form" }
#@markdown Clone the repository and install all dependencies.
import subprocess, sys, os

REPO_URL = "https://github.com/GodL-x-SouL/Captioner-for-Colab.git"
REPO_DIR = "Captioner-for-Colab"

# Clone repo
if not os.path.exists(REPO_DIR):
    print("\033[36mCloning repository...\033[0m")
    subprocess.run(["git", "clone", REPO_URL], check=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("\033[32mRepository cloned.\033[0m")
else:
    print("\033[32mRepository already exists.\033[0m")

# Install dependencies
print("\033[36mInstalling dependencies...\033[0m")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "requests", "flask", "Pillow"],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("\033[32mDependencies installed.\033[0m")

# Install cloudflared
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("\033[36mInstalling cloudflared...\033[0m")
    subprocess.run([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "/usr/local/bin/cloudflared"
    ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"],
                   check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("\033[32mCloudflared installed.\033[0m")
else:
    print("\033[32mCloudflared already installed.\033[0m")

os.chdir(REPO_DIR)
print("\n\033[1;32m\u2713 Setup complete. Proceed to next cell.\033[0m")

In [ ]:
#@title Launch Captioner { display-mode: "form" }
#@markdown Start the captioner server and Cloudflare tunnel.
import subprocess, time, re, threading, os, html as html_mod
from IPython.display import display, HTML, update_display

os.chdir('/content/Captioner-for-Colab')

# Kill any existing processes on port 7860
subprocess.run(["fuser", "-k", "7860/tcp"],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)

# Clear old logs
for f in ['server.log', 'tunnel.log']:
    if os.path.exists(f):
        os.remove(f)

# Start captioner server
server_log = open('server.log', 'w')
server_proc = subprocess.Popen(
    [sys.executable, 'captioner.py'],
    stdout=server_log, stderr=subprocess.STDOUT
)
time.sleep(4)

# Start cloudflare tunnel
tunnel_log = open('tunnel.log', 'w')
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:7860'],
    stdout=tunnel_log, stderr=subprocess.STDOUT
)

# Wait for tunnel URL
tunnel_url = ""
for i in range(45):
    time.sleep(1)
    try:
        with open('tunnel.log', 'r') as f:
            content = f.read()
            match = re.search(r'(https://[a-zA-Z0-9\-]+\.trycloudflare\.com)', content)
            if match:
                tunnel_url = match.group(1)
                break
    except:
        pass

if not tunnel_url:
    tunnel_url = "Tunnel failed to start. Check logs."

# CSS + HTML template
def build_html(t_url, logs_text, server_alive, tunnel_alive):
    esc = html_mod.escape
    url_color = '#16a34a' if 'trycloudflare.com' in t_url else '#dc2626'
    server_dot = '#16a34a' if server_alive else '#dc2626'
    server_label = 'Running' if server_alive else 'Stopped'
    tunnel_dot = '#16a34a' if tunnel_alive else '#dc2626'
    tunnel_label = 'Connected' if tunnel_alive else 'Disconnected'
    log_lines = esc(logs_text[-3500:])
    return f"""
<link href='https://fonts.googleapis.com/css2?family=Playfair+Display:wght@400;600;700&family=Inter:wght@300;400;500&family=JetBrains+Mono:wght@300;400;500&display=swap' rel='stylesheet'>
<style>
  .oc-wrap {{
    max-width:640px; margin:0 auto; padding:24px;
    background:linear-gradient(145deg, #FAF8F5 0%, #F3EDE6 100%);
    border-radius:16px; border:1px solid #E8E0D5;
    box-shadow:0 8px 32px rgba(27,42,74,0.06), 0 1px 3px rgba(0,0,0,0.04);
    font-family:'Inter',sans-serif;
  }}
  .oc-header {{ text-align:center; margin-bottom:24px; }}
  .oc-header h1 {{
    font-family:'Playfair Display',serif; font-size:26px;
    color:#1B2A4A; margin:0; letter-spacing:0.5px; font-weight:600;
  }}
  .oc-header p {{
    font-size:12px; color:#8B7D6B; margin:6px 0 0;
    letter-spacing:1.5px; text-transform:uppercase; font-weight:400;
  }}
  .oc-divider {{
    width:60px; height:0; border-top:1.5px solid #C5A55A;
    margin:14px auto 0;
  }}
  .oc-card {{
    background:#FFFFFF; border:1px solid #E8E0D5;
    border-radius:10px; padding:18px 20px; margin-bottom:14px;
    box-shadow:0 1px 4px rgba(0,0,0,0.03);
  }}
  .oc-label {{
    font-family:'JetBrains Mono',monospace; font-size:9px;
    letter-spacing:2px; text-transform:uppercase;
    color:#A0926E; margin-bottom:10px; font-weight:500;
  }}
  .oc-url-row {{
    display:flex; align-items:center; gap:10px;
    background:#FAF8F5; border:1px solid #E8E0D5;
    border-radius:8px; padding:12px 14px;
  }}
  .oc-url-text {{
    font-family:'JetBrains Mono',monospace; font-size:13px;
    color:{url_color}; flex:1; word-break:break-all;
    text-decoration:none; line-height:1.4;
  }}
  .oc-copy-btn {{
    background:#1B2A4A; color:#FAF8F5; border:none;
    border-radius:6px; padding:8px 16px; cursor:pointer;
    font-family:'JetBrains Mono',monospace; font-size:10px;
    letter-spacing:1px; text-transform:uppercase;
    transition:background 0.2s; white-space:nowrap;
  }}
  .oc-copy-btn:hover {{ background:#2D4A7A; }}
  .oc-status-row {{ display:flex; gap:12px; flex-wrap:wrap; }}
  .oc-status-item {{
    flex:1; min-width:120px; display:flex; align-items:center; gap:8px;
    font-size:12px; color:#4A4A4A;
  }}
  .oc-dot {{
    width:8px; height:8px; border-radius:50%; flex-shrink:0;
  }}
  .oc-logs {{
    background:#1B2A4A; color:#D4CFC7;
    font-family:'JetBrains Mono',monospace; font-size:10px;
    line-height:1.9; padding:14px 16px; border-radius:8px;
    max-height:280px; overflow-y:auto; white-space:pre-wrap;
    word-break:break-word; min-height:80px;
  }}
  .oc-logs::-webkit-scrollbar {{ width:4px; }}
  .oc-logs::-webkit-scrollbar-track {{ background:transparent; }}
  .oc-logs::-webkit-scrollbar-thumb {{ background:#3D5A80; border-radius:2px; }}
  .oc-badge {{
    display:inline-block; padding:3px 10px; border-radius:20px;
    font-family:'JetBrains Mono',monospace; font-size:9px;
    letter-spacing:1px; text-transform:uppercase; font-weight:500;
  }}
  .oc-badge-ok {{ background:rgba(22,163,74,0.1); color:#16a34a; border:1px solid rgba(22,163,74,0.2); }}
  .oc-badge-err {{ background:rgba(220,38,38,0.1); color:#dc2626; border:1px solid rgba(220,38,38,0.2); }}
  .oc-open-link {{
    display:block; text-align:center; margin-top:12px;
    font-family:'Inter',sans-serif; font-size:12px;
    color:#1B2A4A; text-decoration:none; letter-spacing:0.3px;
    padding:10px; border:1px solid #C5A55A; border-radius:8px;
    transition:all 0.2s;
  }}
  .oc-open-link:hover {{ background:#1B2A4A; color:#FAF8F5; border-color:#1B2A4A; }}
</style>
<div class='oc-wrap'>
  <div class='oc-header'>
    <h1>Dataset Captioner</h1>
    <p>Image Captioning Studio</p>
    <div class='oc-divider'></div>
  </div>

  <div class='oc-card'>
    <div class='oc-label'>Tunnel URL</div>
    <div class='oc-url-row'>
      <a class='oc-url-text' href='{esc(t_url)}' target='_blank'>{esc(t_url)}</a>
      <button class='oc-copy-btn' onclick="navigator.clipboard.writeText('{esc(t_url)}').then(()=>{{this.textContent='Copied';setTimeout(()=>{{this.textContent='Copy'}},1500)}})">Copy</button>
    </div>
    <a class='oc-open-link' href='{esc(t_url)}' target='_blank'>\u2192 Open Captioner in New Tab</a>
  </div>

  <div class='oc-card'>
    <div class='oc-label'>Status</div>
    <div class='oc-status-row'>
      <div class='oc-status-item'>
        <div class='oc-dot' style='background:{server_dot}'></div>
        <span>Server</span>
        <span class='oc-badge {'oc-badge-ok' if server_alive else 'oc-badge-err'}'>{server_label}</span>
      </div>
      <div class='oc-status-item'>
        <div class='oc-dot' style='background:{tunnel_dot}'></div>
        <span>Tunnel</span>
        <span class='oc-badge {'oc-badge-ok' if tunnel_alive else 'oc-badge-err'}'>{tunnel_label}</span>
      </div>
    </div>
  </div>

  <div class='oc-card'>
    <div class='oc-label'>Logs</div>
    <div class='oc-logs'>{log_lines if log_lines else 'Awaiting logs...'}</div>
  </div>
</div>
"""

# Initial display
display_id = 'oc-live'
init_logs = "Server starting...\n"
try:
    with open('server.log', 'r') as f:
        init_logs = f.read()
except:
    pass

display(
    HTML(build_html(tunnel_url, init_logs,
                    server_proc.poll() is None,
                    tunnel_proc.poll() is None)),
    display_id=display_id
)

# Background updater
def _update_loop():
    while True:
        time.sleep(3)
        logs = ''
        try:
            with open('server.log', 'r') as f:
                logs = f.read()
        except:
            logs = 'Waiting...'
        srv_alive = server_proc.poll() is None
        tun_alive = tunnel_proc.poll() is None
        try:
            update_display(
                HTML(build_html(tunnel_url, logs, srv_alive, tun_alive)),
                display_id=display_id
            )
        except:
            pass

updater = threading.Thread(target=_update_loop, daemon=True)
updater.start()